# Lesson 07: Gaussian Processes and Active Learning - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: GP uncertainty, UCB, expected improvement, and sequential experiments.

## Learning goals
1. Fit a Gaussian Process to three initial experiments.
2. Compute a UCB acquisition function.
3. Select the next experiment.
4. Run a short active-learning loop.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

rng = np.random.default_rng(7)

def true_process(x):
    return np.sin(x/1.5)*3 + 5 - 0.15*x

X_obs = np.array([[1.0], [4.0], [8.0]])
y_obs = true_process(X_obs.ravel()) + rng.normal(0, 0.15, len(X_obs))
grid = np.linspace(0, 10, 400).reshape(-1, 1)

kernel = ConstantKernel(1.0) * RBF(length_scale=1.2) + WhiteKernel(noise_level=0.05)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)
gp.fit(X_obs, y_obs)
mu, std = gp.predict(grid, return_std=True)
beta = 1.8
ucb = mu + beta*std
next_x = grid[np.argmax(ucb), 0]

fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax[0].plot(grid, true_process(grid.ravel()), "k--", label="true process")
ax[0].plot(grid, mu, label="GP mean")
ax[0].fill_between(grid.ravel(), mu-2*std, mu+2*std, alpha=0.2, label="95% band")
ax[0].scatter(X_obs.ravel(), y_obs, c="tab:blue", zorder=3, label="experiments")
ax[0].axvline(next_x, color="tab:red", ls="--", label="next experiment")
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[1].plot(grid, ucb, color="tab:red", label="UCB")
ax[1].axvline(next_x, color="tab:red", ls="--")
ax[1].set_xlabel("Residence time [min]")
ax[1].set_ylabel("Acquisition")
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Next experiment by UCB: x = {next_x:.2f}")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?